In [41]:
%load_ext autoreload
%autoreload 2


import importlib
import math
import random

### My libraries (not pip)

import Mathlib
import Activation
import Loss
import Optimizer
import Accuracy
import Neuron
import Layer
import Batch

importlib.reload(Mathlib)
importlib.reload(Activation)
importlib.reload(Loss)
importlib.reload(Optimizer)
importlib.reload(Accuracy)
importlib.reload(Neuron)
importlib.reload(Layer)
importlib.reload(Batch)

from Neuron import Neuron
from Layer import Layer
from Batch import Batch


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Go to `proofs_math/ActivationFuncs` for all Activation function proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/ActivationFuncs/SoftmaxDerivative.jpg?version=1"
         alt="Softmax Proof"
         width="500">
  </div>

  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/ActivationFuncs/ReLUDerivative.jpg?version=1"
         alt="ReLU Proof"
         width="500">
  </div>

</div>



#### Go to `proofs_math/Loss` for all Loss proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Loss/SoftmaxCrossEntropyDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/Loss/LossDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>

#### Go to `proofs_math/Neuron` for all Neuron proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Neuron/NeuronDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>



## Function that will be used to create datasets

In [16]:
def createInputs(inputCount=10):
    return([Mathlib.randomNumber() for _ in range(inputCount)])

## Functions to run a forward and backward pass on a super simple network with two layer totaling 9 neurons

In [48]:
random.seed(1)

inputsCount = 5
target = 2
inputs = createInputs(inputsCount)

layer1NeuronCount = 4
layer2NeuronCount = 5

layer1 = Layer(inputsCount, layer1NeuronCount, activationFunc=Activation.ReLU)         #< weights/biases determine neuron count
layer2 = Layer(layer1NeuronCount, layer2NeuronCount, activationFunc=Activation.ReLU)   #< weights/biases determine neuron count

accuracyCalculator = Accuracy.Accuracy_hard()

def stepForward():
    layer1Output = layer1.forward(inputs)
    layer2Output = layer2.forward(layer1Output)
    softmaxOutput = Activation.Softmax.forward(layer2Output)
    error = Loss.Entropy.forwardSparse(softmaxOutput, target)
    accuracy = accuracyCalculator.epoch(softmaxOutput, target)    #< we want to maximize the 2nd output
    return(layer1Output, layer2Output, softmaxOutput, error, accuracy)

def stepBackward(softmaxOutput):
    d_error  = Loss.Entropy.backwardSparse(softmaxOutput, target)
    d_softmax = Activation.Softmax.backward(softmaxOutput)
    d_crossEntropy = Mathlib.dotVectorMatrix(d_error, d_softmax) #< not that we can calculate the cross entropy function much simpler using the shortcut backwardCrossEntropy, but I choose to take the long way in this example because its easier to understand
    d_layer2 = layer2.backward(d_crossEntropy)
    d_layer1 = layer1.backward(d_layer2)
    return(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1)

def printForwardSummary(layer1Output, layer2Output, softmaxOutput, error, accuracy):
    print(f"INPUTS: {inputs}\n    This layer should have {inputsCount} inputs")
    print(f"WEIGHTS: {layer1.getWeights()}\n    this layer should have {layer1NeuronCount} weights with {inputsCount} values in them")
    print(f"BIASES: {layer1.getBiases()}\n    this layer should have {layer1NeuronCount} biases")
    print(f"RESULT: {layer1Output}\n     this layer should have {layer1NeuronCount} outputs")
    print()
    print(f"WEIGHTS: {layer1.getWeights()}\n    this layer should have {layer2NeuronCount} weights with {len(layer1Output)} values in them")
    print(f"BIASES: {layer1.getBiases()}\n    this layer should have {layer2NeuronCount} biases")
    print(f"RESULT: {layer2Output}\n     this layer should have {layer2NeuronCount} outputs")
    print("SOFTMAX: ", softmaxOutput)
    print("TARGET: ", target)
    print(f"ERROR: {error} should be between 0 and 16.12")
    print("ACCURACY: ", accuracy)

def printBackwardSummary(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1):
    print("d_ERROR: ", d_error)
    print(f"d_SOFTMAX: {d_softmax}\n    this layer should be {layer2NeuronCount}x{len(d_softmax)}")
    print(f"d_CROSS_ENTROPY: {d_crossEntropy}\n    this layer should have {layer2NeuronCount} outputs")
    print("d_LAYER 2: ", d_layer2)
    print("d_LAYER 1: ", d_layer1)

layer1Output, layer2Output, softmaxOutput, error, accuracy = stepForward()
printForwardSummary(layer1Output, layer2Output, softmaxOutput, error, accuracy)
d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1 = stepBackward(softmaxOutput)
printBackwardSummary(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1)

INPUTS: [-7.31, 6.95, 5.28, -4.9, -0.09]
    This layer should have 5 inputs
WEIGHTS: [[-0.1, 0.3, 0.58, -0.81, -0.94], [0.67, -0.13, 0.52, -1.0, -0.11], [0.44, -0.54, 0.89, 0.8, -0.94], [-0.95, 0.08, 0.88, -0.24, -0.57]]
    this layer should have 4 weights with 5 values in them
BIASES: [-1.56, -9.42, -5.57, -1.24]
    this layer should have 4 biases
RESULT: [8.372, 0.0, 0.0, 12.134199999999998]
     this layer should have 4 outputs

WEIGHTS: [[-0.1, 0.3, 0.58, -0.81, -0.94], [0.67, -0.13, 0.52, -1.0, -0.11], [0.44, -0.54, 0.89, 0.8, -0.94], [-0.95, 0.08, 0.88, -0.24, -0.57]]
    this layer should have 5 weights with 4 values in them
BIASES: [-1.56, -9.42, -5.57, -1.24]
    this layer should have 5 biases
RESULT: [0.0, 3.6514959999999994, 14.683777999999998, 19.016888, 18.444812]
     this layer should have 5 outputs
SOFTMAX:  [3.492261479035917e-09, 1.3456475252761318e-07, 0.008321287218754293, 0.6339226581263162, 0.35775591659791545]
TARGET:  2
ERROR: 4.788938322323381 should be bet

### Optimizing the neural network

In [50]:
optimizer = Optimizer.Optimizer_SGD(learning_rate=0.0001)
for i in range(5000):
    layer1Output, layer2Output, softmaxOutput, error, accuracy = stepForward()
    d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1 = stepBackward(softmaxOutput)
    optimizer.update(layer2)
    optimizer.update(layer1)
    print(f"Epoch {i+1}: Error: {error}, accuracy: {accuracy}")

Epoch 1: Error: 0.00415417603387223, accuracy: 0.9772091163534586
Epoch 2: Error: 0.0041532759419345994, accuracy: 0.9772136717969219
Epoch 3: Error: 0.004152376231464924, accuracy: 0.9772182254196643
Epoch 4: Error: 0.004151476902222373, accuracy: 0.9772227772227772
Epoch 5: Error: 0.00415057795396645, accuracy: 0.9772273272073512
Epoch 6: Error: 0.004149679386457665, accuracy: 0.9772318753744758
Epoch 7: Error: 0.004148781199455636, accuracy: 0.9772364217252396
Epoch 8: Error: 0.004147883392720764, accuracy: 0.9772409662607306
Epoch 9: Error: 0.0041469859660136724, accuracy: 0.9772455089820359
Epoch 10: Error: 0.0041460889190950995, accuracy: 0.9772500498902414
Epoch 11: Error: 0.004145192251725893, accuracy: 0.9772545889864326
Epoch 12: Error: 0.004144295963667019, accuracy: 0.9772591262716936
Epoch 13: Error: 0.004143400054679885, accuracy: 0.9772636617471081
Epoch 14: Error: 0.0041425045245262375, accuracy: 0.9772681954137588
Epoch 15: Error: 0.0041416093729673776, accuracy: 0.977

In [81]:
import json
with open("datasets/spiralData.json", "r") as f:  #< Data taken from Sentdex's nnfs library. https://nnfs.io/
    a = json.load(f)
    X = a["X"]
    y = a["y"]

spiralLayer1 = Batch(
# spiralLayer1 = Layer(
                     inputCount=2,
                     neuronCount=64,   #< output features
                     activationFunc=Activation.LeakyReLU)
spiralLayer2 = Batch(
# spiralLayer2 = Layer(
                     inputCount=64,
                     neuronCount=3,   #< output features
                     activationFunc=Activation.Pass)
softmaxCrossEntropy = Loss.SoftmaxCrossEntropy()
spiralOptimizer = Optimizer.Optimizer_SGD(learning_rate=0.05)
accuracyCalculator = Accuracy.Accuracy_hard()

for epoch in range(10_001):
    ### Forwards pass
    ### layer1 -> layer2 -> softmax_error
    layer1Output = spiralLayer1.forward(X)                                    #< returns a list of lists of outputs
    layer2Output = spiralLayer2.forward(layer1Output)                         #< returns a list of lists of outputs
    error = softmaxCrossEntropy.forward_batch(layer2Output, y)                #< returns mean error
    # error = softmaxCrossEntropy.forward(layer2Output, y)                    #< returns mean error
    accuracy = accuracyCalculator.epoch(layer2Output, y)                      #< returns accuracy of model %right

    ### Backwards pass
    ### error_softmax -> layer2 ->  layer1 
    d_crossEntropy = softmaxCrossEntropy.backward_batch()
    # d_crossEntropy = softmaxCrossEntropy.backward()
    d_layer2 = spiralLayer2.backward(d_crossEntropy)
    # print([Mathlib.mean(d) for d in d_layer2])
    d_layer1 = spiralLayer1.backward(d_layer2)
    # print([Mathlib.mean(d) for d in d_layer1])

    spiralOptimizer.update(spiralLayer2)
    spiralOptimizer.update(spiralLayer1)
    if epoch%100 == 0:
        print(f"Epoch {epoch}: Error: {error}, accuracy: {accuracy}")

Epoch 0: Error: 1.1079121473327782, accuracy: 0.29333333333333333
Epoch 100: Error: 1.0768375121064055, accuracy: 0.4013861386138614
Epoch 200: Error: 1.0700412756793751, accuracy: 0.4241625207296849
Epoch 300: Error: 1.0677396650082465, accuracy: 0.4328239202657807
Epoch 400: Error: 1.0665636320855938, accuracy: 0.4350041562759767
Epoch 500: Error: 1.0657726258280846, accuracy: 0.4355954757152362
Epoch 600: Error: 1.0651664309283504, accuracy: 0.4359012756516916
Epoch 700: Error: 1.0646813286384726, accuracy: 0.43606276747503564
Epoch 800: Error: 1.0642838913091321, accuracy: 0.43462754889721184
Epoch 900: Error: 1.0639535746218087, accuracy: 0.4325009248982612
Epoch 1000: Error: 1.0636763446999786, accuracy: 0.4305994005994006
Epoch 1100: Error: 1.0634409895999148, accuracy: 0.4295246745382985
Epoch 1200: Error: 1.0632387205520657, accuracy: 0.4287316125451013
Epoch 1300: Error: 1.0630630818697608, accuracy: 0.42806046630796823
Epoch 1400: Error: 1.0629085988871472, accuracy: 0.42755

In [69]:
from nnfs.datasets import spiral_data
X, y = spiral_data(samples=100, classes=3)
len(X)

300